In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("OLLAMA_BASE_URL")

REPO_ROOT = Path.cwd().parent  # adjust if your notebook isn't directly inside notebooks/

sys.path.insert(0, str(REPO_ROOT / "src"))

from wraith.parsers.cowrie_parser import parse_sessions
from wraith.llm.session_assessment import analyze_session
from wraith.enrichment.triage import triage_batch
from wraith.audit.audit_log import log_batch

## Parse raw Cowrie logs

Deterministic layer only - no LLM calls happen here. `parse_sessions()` groups
raw events by session and returns a list of `SessionSummary` objects.

In [2]:
SAMPLE_LOG_PATH = REPO_ROOT / "data" / "raw" / "sample_cowrie_logs.json"

summaries = parse_sessions(str(SAMPLE_LOG_PATH))
print(f"Parsed {len(summaries)} session(s)")
for s in summaries:
	print(f"  {s.session_id}  src_ip={s.src_ip}  attempts={s.login_attempts}  "
	      f"success={s.login_success}  commands={len(s.commands)}  downloads={len(s.files_downloaded)}")

Parsed 4 session(s)
  a1b2c3d4  src_ip=185.220.101.45  attempts=0  success=False  commands=0  downloads=0
  e5f6a7b8  src_ip=45.155.204.13  attempts=5  success=True  commands=0  downloads=0
  c9d0e1f2  src_ip=185.220.101.45  attempts=1  success=True  commands=6  downloads=1
  0f1e2d3c  src_ip=103.42.88.201  attempts=0  success=False  commands=0  downloads=0


## Run triage

`triage_batch()` skips the LLM entirely for trivial connect/disconnect sessions,
sends everything else to `analyze_session()` (Ollama, candidate-constrained schema,
guardrail-validated), and runs IoC correlation on anything that comes back
suspicious/malicious.

In [3]:
results = triage_batch(summaries, analyze_session, BASE_URL, MODEL)

for r in results:
	print(f"\n--- {r['session_id']} ({r['method']}) ---")
	print(f"classification={r['classification']}  confidence={r['confidence']}")
	if r["assessment"]:
		print(r["assessment"].model_dump_json(indent=2))
	if r["ioc_history"]:
		ioc = r["ioc_history"]
		print(f"IoC correlation for {ioc.indicator}: {len(ioc.related_session_ids)} related "
		      f"session(s) ({ioc.related_session_ids}), any_login_success={ioc.any_login_success}, "
		      f"total_commands={ioc.total_commands}, total_downloads={ioc.total_downloads}")

[c9d0e1f2] Note: model explicitly excluded these candidates:
  T1078: Although the attacker successfully authenticated after only one attempt, this is not a significant indicator of malicious activity in isolation.

--- a1b2c3d4 (rule-based (trivial session, LLM skipped)) ---
classification=benign  confidence=high

--- e5f6a7b8 (LLM assessment) ---
classification=suspicious  confidence=medium
{
  "summary": "An SSH connection was made from IP address 45.155.204.13 with 5 login attempts before successfully authenticating as 'root:raspberry'. The duration of the session was approximately 15 seconds.",
  "classification": "suspicious",
  "confidence": "medium",
  "techniques": [
    {
      "technique_id": "T1110.001",
      "included": true,
      "justification": "The attacker made 5 login attempts before successfully authenticating, which is above the threshold of 3 attempts."
    }
  ]
}
IoC correlation for 45.155.204.13: 0 related session(s) ([]), any_login_success=False, total_comma

## Write the audit trail

Every result (rule-based skip or LLM assessment) gets appended to the JSONL
audit trail. `log_batch()` also prints anything the contextually-blind
heuristic auto-flags for review (e.g. an excluded technique justified with
isolation-language on a session that has other included techniques) as it writes.

In [4]:
records = log_batch(results, model=MODEL, seed=42)

flagged = [r for r in records if r["flagged_for_review"]]
print(f"\n{len(flagged)} of {len(records)} record(s) flagged for review")


1 record(s) flagged for review:
  [c9d0e1f2] T1078 excluded using isolation-type language ('Although the attacker successfully authenticated after only one attempt, this is not a significant indicator of malicious activity in isolation.') despite 4 other technique(s) included in the same session

1 of 4 record(s) flagged for review


## Inspect the audit trail

Quick tabular view of what actually landed in the JSONL file this run - useful
for spot-checking classifications/flags without opening the raw log.

In [6]:
import pandas as pd

df = pd.DataFrame(records)
df[["session_id", "src_ip", "classification", "confidence", "method", "flagged_for_review"]]

,session_id,src_ip,classification,confidence,method,flagged_for_review
0,a1b2c3d4,185.220.101.45,benign,high,"rule-based (trivial session, LLM skipped)",False
1,e5f6a7b8,45.155.204.13,suspicious,medium,LLM assessment,False
2,c9d0e1f2,185.220.101.45,suspicious,high,LLM assessment,True
3,0f1e2d3c,103.42.88.201,benign,high,"rule-based (trivial session, LLM skipped)",False
